# Electrolyzer Patent Analysis - Enhanced Interactive Sector Analysis

### Advanced version with sector classification, search functionality and interactive aggregation

This notebook provides a comprehensive analysis of electrolyzer patent applicants with enhanced features:

**Key Features:**
- Enhanced sector classification (Companies, Universities, Research Institutions, etc.)
- Prioritizes PSN_SECTOR from database, with smart fallback for vague values
- Color-coded visualizations based on applicant sectors
- Interactive HTML file with pre-loaded search functionality
- User-controlled aggregation through selection interface
- Downloadable files (HTML and Excel) with embedded visualizations
- Smart search allowing keyword-based applicant grouping
- Real-time aggregation of patent families for selected applicants

**Data source**: Electrolyzer Enhanced Final Dataset 2025 with docdb_family_id values
**Enhanced feature**: psn_sector information from TLS206_PERSON table
**Priority**: Database psn_sector classification > Name-based fallback (except for vague values)

In [1]:
# Import libraries
from epo.tipdata.patstat import PatstatClient
from epo.tipdata.patstat.database.models import TLS201_APPLN, TLS206_PERSON, TLS207_PERS_APPLN

import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.io as pio
from sqlalchemy import func
from datetime import datetime
import re
import warnings
warnings.filterwarnings('ignore')

# Connect to PATSTAT TIP database
patstat = PatstatClient(env='PROD')
db = patstat.orm()

print("✅ Libraries and PATSTAT connection initialized")
print(f"🕐 Analysis started at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

✅ Libraries and PATSTAT connection initialized
🕐 Analysis started at: 2026-03-12 15:04:00


In [2]:
# ==============================================
# LOAD ELECTROLYZER DATASET
# ==============================================

# Load electrolyzer dataset to get docdb_family_id values
print("📊 Loading electrolyzer dataset...")
electrolyzer_df = pd.read_excel('03_Dataset_Enhancement__Elettrolizzatori_Enhanced_Final_Dataset_2025!!!.xlsx')
print(f"✅ Loaded {len(electrolyzer_df)} electrolyzer patents")

# Extract unique docdb_family_id values
electrolyzer_family_ids = electrolyzer_df['docdb_family_id'].unique().tolist()
print(f"📋 Found {len(electrolyzer_family_ids)} unique patent families")
print(f"🔍 Sample family IDs: {electrolyzer_family_ids[:10]}")

# Configuration parameters
START_YEAR = 2000
END_YEAR = 2023
BATCH_SIZE_YEARS = 3

print(f"\n⚙️ Configuration:")
print(f"   📅 Year range: {START_YEAR}-{END_YEAR}")
print(f"   🔢 Patent families to process: {len(electrolyzer_family_ids)}")
print(f"   📝 Mode: Enhanced extraction with sector information")

📊 Loading electrolyzer dataset...
✅ Loaded 18811 electrolyzer patents
📋 Found 18811 unique patent families
🔍 Sample family IDs: [3819525, 3824194, 3832658, 4168002, 4168863, 4169054, 4169160, 4170032, 4581634, 4583816]

⚙️ Configuration:
   📅 Year range: 2000-2023
   🔢 Patent families to process: 18811
   📝 Mode: Enhanced extraction with sector information


In [3]:
# ==============================================
# ENHANCED DATA EXTRACTION WITH SECTOR INFO
# ==============================================

def extract_batch_data_enhanced(start_year, end_year):
    """
    Extract enhanced patent applicant data with sector information for a specific year range
    """
    print(f"  🔍 Processing years {start_year}-{end_year}...")
    
    try:
        # Filter electrolyzer families that fall within the year range
        family_ids_in_range = (
            db.query(TLS201_APPLN.docdb_family_id)
            .filter(
                TLS201_APPLN.docdb_family_id.in_(electrolyzer_family_ids),
                TLS201_APPLN.earliest_filing_year.between(start_year, end_year)
            )
            .distinct()
        ).all()
        
        family_ids_list = [row[0] for row in family_ids_in_range]
        
        if not family_ids_list:
            print(f"    ⚠️ No electrolyzer patents found for {start_year}-{end_year}")
            return pd.DataFrame()
        
        print(f"    📋 Found {len(family_ids_list)} electrolyzer families in range")
        
        # Enhanced query with sector information from TLS206_PERSON
        query = db.query(
            TLS206_PERSON.psn_name,
            TLS206_PERSON.psn_sector,  # This is the key enhancement!
            TLS201_APPLN.docdb_family_id,
            TLS201_APPLN.earliest_filing_year,
            TLS207_PERS_APPLN.person_id
        ).join(
            TLS207_PERS_APPLN, TLS206_PERSON.person_id == TLS207_PERS_APPLN.person_id
        ).join(
            TLS201_APPLN, TLS207_PERS_APPLN.appln_id == TLS201_APPLN.appln_id
        ).filter(
            TLS207_PERS_APPLN.applt_seq_nr != 0,
            TLS201_APPLN.docdb_family_id.in_(family_ids_list),
            TLS201_APPLN.earliest_filing_year.between(start_year, end_year)
        ).order_by(
            TLS206_PERSON.psn_name,
            TLS201_APPLN.docdb_family_id
        )
        
        result = query.all()
        
        if result:
            df = pd.DataFrame(result, columns=['Applicant_Name', 'PSN_Sector', 'Patent_Family_ID', 'Filing_Year', 'Person_ID'])
            print(f"    ✅ Found {len(df)} applicant-patent relationships with sector info")
            return df
        else:
            print(f"    ⚠️ No applicant data found for {start_year}-{end_year}")
            return pd.DataFrame()
            
    except Exception as e:
        print(f"    ❌ Error processing {start_year}-{end_year}: {str(e)}")
        return pd.DataFrame()

# Extract data in batches and combine
print("🚀 Starting enhanced electrolyzer patent data extraction...")
all_batches = []

for year in range(START_YEAR, END_YEAR + 1, BATCH_SIZE_YEARS):
    batch_end = min(year + BATCH_SIZE_YEARS - 1, END_YEAR)
    df_batch = extract_batch_data_enhanced(year, batch_end)
    if not df_batch.empty:
        all_batches.append(df_batch)

if all_batches:
    print(f"\n📊 Combining {len(all_batches)} data batches...")
    raw_data = pd.concat(all_batches, ignore_index=True)
    
    print(f"✅ Enhanced data extraction complete!")
    print(f"   📈 Total applicant-patent relationships: {len(raw_data)}")
    print(f"   🏢 Unique applicant names: {raw_data['Applicant_Name'].nunique()}")
    print(f"   🏭 Unique patent families: {raw_data['Patent_Family_ID'].nunique()}")
    print(f"   📅 Year range in data: {raw_data['Filing_Year'].min()}-{raw_data['Filing_Year'].max()}")
    print(f"   🏷️ Applicants with sector info: {raw_data['PSN_Sector'].notna().sum()} / {len(raw_data)}")
    
else:
    print("❌ No data found! Please check the electrolyzer family IDs and year range.")
    raw_data = pd.DataFrame()

🚀 Starting enhanced electrolyzer patent data extraction...
  🔍 Processing years 2000-2002...
    📋 Found 695 electrolyzer families in range
    ✅ Found 1867 applicant-patent relationships with sector info
  🔍 Processing years 2003-2005...
    📋 Found 831 electrolyzer families in range
    ✅ Found 2073 applicant-patent relationships with sector info
  🔍 Processing years 2006-2008...
    📋 Found 1034 electrolyzer families in range
    ✅ Found 3296 applicant-patent relationships with sector info
  🔍 Processing years 2009-2011...
    📋 Found 1298 electrolyzer families in range
    ✅ Found 3868 applicant-patent relationships with sector info
  🔍 Processing years 2012-2014...
    📋 Found 1508 electrolyzer families in range
    ✅ Found 3222 applicant-patent relationships with sector info
  🔍 Processing years 2015-2017...
    📋 Found 2685 electrolyzer families in range
    ✅ Found 4603 applicant-patent relationships with sector info
  🔍 Processing years 2018-2020...
    📋 Found 3492 electrolyz

In [4]:
# ==============================================
# ENHANCED SECTOR CLASSIFICATION FUNCTION
# ==============================================

def classify_applicant_sector(name, psn_sector=None):
    """
    Classify applicant sector using database psn_sector first, then enhanced name-based fallback
    Priority: Database psn_sector classification > Enhanced name-based fallback
    Special handling: Use name-based classification for vague psn_sector values like 'OTHER' and 'UNKNOWN'
    """
    # First priority: Use database psn_sector if available and not vague
    if pd.notna(psn_sector) and psn_sector.strip():
        psn_upper = psn_sector.upper()
        
        # Skip vague database classifications and use name-based instead
        vague_classifications = ['OTHER', 'UNKNOWN', 'UNDEFINED', 'NOT SPECIFIED', 'N/A']
        if psn_upper not in vague_classifications:
            sector_map = {
                'COMPANY': 'Company',
                'INDIVIDUAL': 'Individual',
                'UNIVERSITY': 'University',
                'GOV NON-PROFIT': 'Government/Non-Profit',
                'GOVERNMENT': 'Government/Non-Profit',
                'HOSPITAL': 'Healthcare/Hospital',
                'RESEARCH INSTITUTION': 'Research Institution',
                'RESEARCH': 'Research Institution'
            }
            mapped_sector = sector_map.get(psn_upper)
            if mapped_sector is not None:
                return mapped_sector
        # If psn_sector is vague or unmapped, fall through to name-based classification
    
    # Enhanced name-based classification (used as fallback or for vague PSN_SECTOR values)
    if pd.isna(name) or not name.strip():
        return 'Unknown'
    
    name_upper = name.upper()
    
    # Academy of Sciences and similar research institutions (highest priority)
    academy_keywords = [
        'ACADEMY OF SCIENCES', 'CHINESE ACADEMY', 'ROYAL SOCIETY', 'NATIONAL ACADEMY',
        'ACADEMIA SINICA', 'KOREAN ACADEMY', 'RUSSIAN ACADEMY', 'KOREAN ACADEMY OF SCIENCE'
    ]
    
    # Research institution indicators (comprehensive)
    research_keywords = [
        'RESEARCH INSTITUTE', 'RESEARCH CENTER', 'RESEARCH CENTRE', 'LABORATORY',
        'NATIONAL INSTITUTE', 'FRAUNHOFER', 'CNRS', 'RIKEN', 'MAX PLANCK',
        'RESEARCH FOUNDATION', 'SCIENTIFIC RESEARCH', 'TECHNOLOGY RESEARCH',
        'CLEAN ENERGY RESEARCH', 'ENERGY RESEARCH', 'THERMAL POWER RESEARCH',
        'POWER RESEARCH INSTITUTE', 'RESEARCH AND DEVELOPMENT', 'R&D CENTER'
    ]
    
    # University indicators
    university_keywords = [
        'UNIVERSITY', 'UNIVERSITE', 'UNIVERSITEIT', 'UNIVERSIDAD', 'UNIVERSITAET',
        'COLLEGE', 'SCHOOL OF', 'TECHNICAL UNIVERSITY', 'POLYTECHNIC UNIVERSITY',
        'INSTITUTE OF TECHNOLOGY', 'TECHNICAL INSTITUTE'
    ]
    
    # Government/Public indicators
    government_keywords = [
        'MINISTRY', 'GOVERNMENT', 'STATE GRID', 'NATIONAL', 'STATE', 'FEDERAL', 'PUBLIC',
        'COMMISSARIAT', 'AGENCY', 'ADMINISTRATION', 'DEPARTMENT', 'MUNICIPAL',
        'PROVINCIAL', 'REGIONAL'
    ]
    
    # Healthcare indicators
    healthcare_keywords = ['HOSPITAL', 'MEDICAL', 'HEALTH', 'CLINIC']
    
    # Individual indicators (enhanced)
    individual_indicators = [',', 'DR.', 'PROF.', 'MR.', 'MS.', 'MRS.']
    if (len(name.split()) <= 3 and 
        any(indicator in name_upper for indicator in individual_indicators)):
        return 'Individual'
    
    # Check categories in priority order
    # 1. Academy of Sciences (highest priority for research classification)
    if any(keyword in name_upper for keyword in academy_keywords):
        return 'Research Institution'
    
    # 2. Research institutions (before universities to catch research-focused institutes)
    elif any(keyword in name_upper for keyword in research_keywords):
        return 'Research Institution'
    
    # 3. Universities
    elif any(keyword in name_upper for keyword in university_keywords):
        return 'University'
    
    # 4. Government/Public entities
    elif any(keyword in name_upper for keyword in government_keywords):
        return 'Government/Non-Profit'
    
    # 5. Healthcare
    elif any(keyword in name_upper for keyword in healthcare_keywords):
        return 'Healthcare/Hospital'
    
    # 6. Default to Company
    else:
        return 'Company'

print("✅ Enhanced sector classification function defined successfully!")
print("🔍 Features:")
print("   • Prioritizes database PSN_SECTOR when reliable")
print("   • Falls back to name-based classification for vague values ('OTHER', 'UNKNOWN')")
print("   • Enhanced detection of Research Institutions and Academies of Sciences")
print("   • Improved keyword matching for all sectors")

✅ Enhanced sector classification function defined successfully!
🔍 Features:
   • Prioritizes database PSN_SECTOR when reliable
   • Falls back to name-based classification for vague values ('OTHER', 'UNKNOWN')
   • Enhanced detection of Research Institutions and Academies of Sciences
   • Improved keyword matching for all sectors


In [5]:
# ==============================================
# APPLY SECTOR CLASSIFICATION
# ==============================================

if not raw_data.empty:
    print("🏷️ Applying enhanced sector classification...")
    
    # Apply sector classification using the enhanced function
    raw_data['Sector'] = raw_data.apply(
        lambda row: classify_applicant_sector(row['Applicant_Name'], row['PSN_Sector']), 
        axis=1
    )
    
    # Count classification sources
    db_classified = raw_data['PSN_Sector'].notna().sum()
    name_classified = raw_data['PSN_Sector'].isna().sum()
    vague_psn = raw_data[raw_data['PSN_Sector'].isin(['OTHER', 'UNKNOWN', 'UNDEFINED'])]['PSN_Sector'].count()
    
    print(f"✅ Enhanced sector classification complete!")
    print(f"   🗄️ Database sector info: {db_classified} records")
    print(f"   📝 Name-based classification: {name_classified} records")
    print(f"   🔄 Vague PSN_SECTOR values (using name-based): {vague_psn} records")
    
    # Show sector distribution
    sector_counts = raw_data['Sector'].value_counts()
    print(f"\n📊 Sector distribution:")
    for sector, count in sector_counts.items():
        percentage = (count / len(raw_data)) * 100
        print(f"   {sector}: {count:,} ({percentage:.1f}%)")
        
    # Show examples of improved classification
    print(f"\n🔍 Examples of sector classification:")
    examples = [
        'CHINESE ACADEMY OF SCIENCES',
        'HUANENG CLEAN ENERGY RESEARCH INSTITUTE', 
        'TSINGHUA UNIVERSITY',
        'HONDA MOTOR COMPANY'
    ]
    
    for example in examples:
        matches = raw_data[raw_data['Applicant_Name'] == example]
        if not matches.empty:
            sector = matches.iloc[0]['Sector']
            psn_sector = matches.iloc[0]['PSN_Sector']
            print(f"   📌 {example[:50]}... → {sector} (DB: {psn_sector})")
else:
    print("⚠️ No data available for sector classification.")

🏷️ Applying enhanced sector classification...
✅ Enhanced sector classification complete!
   🗄️ Database sector info: 36806 records
   📝 Name-based classification: 0 records
   🔄 Vague PSN_SECTOR values (using name-based): 5931 records

📊 Sector distribution:
   Company: 22,181 (60.3%)
   Individual: 8,374 (22.8%)
   University: 4,198 (11.4%)
   Government/Non-Profit: 1,182 (3.2%)
   Research Institution: 769 (2.1%)
   Healthcare/Hospital: 102 (0.3%)

🔍 Examples of sector classification:
   📌 CHINESE ACADEMY OF SCIENCES... → Research Institution (DB: GOV NON-PROFIT UNIVERSITY)
   📌 HUANENG CLEAN ENERGY RESEARCH INSTITUTE... → Company (DB: COMPANY)
   📌 TSINGHUA UNIVERSITY... → University (DB: UNIVERSITY)
   📌 HONDA MOTOR COMPANY... → Company (DB: COMPANY)


In [6]:
# ==============================================
# AGGREGATED APPLICANT SUMMARY WITH SECTORS
# ==============================================

if not raw_data.empty:
    print("📊 Creating enhanced applicant summary with sector information...")
    
    # Aggregate data by applicant name
    applicant_summary = raw_data.groupby(['Applicant_Name', 'Sector']).agg({
        'Patent_Family_ID': 'nunique',  # Count unique patent families
        'Filing_Year': ['min', 'max'],   # Get year range
        'Person_ID': 'nunique'          # Count unique person IDs
    }).reset_index()
    
    # Flatten column names
    applicant_summary.columns = ['Applicant_Name', 'Sector', 'Patent_Families', 'First_Year', 'Last_Year', 'Person_IDs']
    
    # Sort by number of patent families (descending)
    applicant_summary = applicant_summary.sort_values('Patent_Families', ascending=False).reset_index(drop=True)
    
    # Add rank
    applicant_summary['Rank'] = range(1, len(applicant_summary) + 1)

    # Build per-applicant list of unique patent family IDs.
    # Embedded in the HTML so the browser can compute a deduplicated total
    # and detect co-applicant overlaps when multiple applicants are selected.
    fam_lists = (
        raw_data.groupby(['Applicant_Name', 'Sector'])['Patent_Family_ID']
        .apply(lambda x: sorted(int(v) for v in x.unique()))
        .reset_index()
        .rename(columns={'Patent_Family_ID': 'Family_ID_List'})
    )
    applicant_summary = applicant_summary.merge(
        fam_lists, on=['Applicant_Name', 'Sector'], how='left'
    )
    applicant_summary['Family_ID_List'] = applicant_summary['Family_ID_List'].apply(
        lambda x: x if isinstance(x, list) else []
    )
    
    print(f"✅ Enhanced summary created!")
    print(f"   🏢 Total unique applicants: {len(applicant_summary)}")
    print(f"   📊 Patent families range: {applicant_summary['Patent_Families'].min()}-{applicant_summary['Patent_Families'].max()}")
    print(f"   🏷️ Sectors identified: {applicant_summary['Sector'].nunique()}")
    

    # ── WARNING: applicants split across multiple sector labels ──────────────
    # Because applicant_summary is grouped by (Applicant_Name, Sector), the same
    # real-world entity can appear more than once if PATSTAT assigns different
    # psn_sector values across its patent records.  When this happens the family
    # count shown on each card in the HTML reflects only one (Name, Sector) row,
    # while the co-applicant overlap table merges all rows by name and may report
    # a slightly higher total.  This faithfully mirrors the original PATSTAT
    # classification and is NOT corrected automatically, but users should verify
    # the sector labels for any flagged applicant before drawing conclusions.
    multi_sector = (
        applicant_summary.groupby('Applicant_Name')['Sector']
        .nunique()
        .reset_index()
        .rename(columns={'Sector': 'n_sectors'})
    )
    multi_sector = multi_sector[multi_sector['n_sectors'] > 1].sort_values('n_sectors', ascending=False)
    if not multi_sector.empty:
        print(f"\n⚠️  DATA QUALITY WARNING – {len(multi_sector)} applicant(s) appear under MORE THAN ONE sector label in PATSTAT.")
        print(   "   This causes a minor discrepancy between the per-card family count in the HTML")
        print(   "   and the 'Total' column in the co-applicant overlap table (which merges all rows).")
        print(   "   Please review these applicants and verify their sector classification:")
        for _, row in multi_sector.iterrows():
            name = row['Applicant_Name']
            sectors = applicant_summary.loc[
                applicant_summary['Applicant_Name'] == name, ['Sector', 'Patent_Families']
            ].to_dict('records')
            sector_str = ', '.join(f"{s['Sector']} ({s['Patent_Families']} fam.)" for s in sectors)
            print(f"   • {name}: {sector_str}")
    else:
        print("\n✅ No applicants with conflicting sector labels found.")
    # ─────────────────────────────────────────────────────────────────────────
    # Show final sector distribution from summary
    final_sector_counts = applicant_summary.groupby('Sector')['Patent_Families'].sum().sort_values(ascending=False)
    total_families = final_sector_counts.sum()
    print(f"\n📊 Final Sector Distribution in Summary:")
    for sector, count in final_sector_counts.items():
        percentage = (count / total_families) * 100
        print(f"   {sector}: {count:,} families ({percentage:.1f}%)")
    
else:
    print("⚠️ No data available for analysis.")
    applicant_summary = pd.DataFrame()

📊 Creating enhanced applicant summary with sector information...
✅ Enhanced summary created!
   🏢 Total unique applicants: 12849
   📊 Patent families range: 1-308
   🏷️ Sectors identified: 6

⚠️  DATA QUALITY WARNING – 49 applicant(s) appear under MORE THAN ONE sector label in PATSTAT.
   This causes a minor discrepancy between the per-card family count in the HTML
   and the 'Total' column in the co-applicant overlap table (which merges all rows).
   Please review these applicants and verify their sector classification:
   • ALLIANCE MAGNESIUM: Company (2 fam.), Individual (2 fam.)
   • AUGE II WAYNE K: Individual (3 fam.), Company (1 fam.)
   • AVRIL: Individual (1 fam.), Company (1 fam.)
   • DALIAN SHUANGDI CREATIVE TECHNOLOGY RESEARCH INSTITUTE COMPANY: Company (10 fam.), Government/Non-Profit (1 fam.)
   • DALIAN SHUANGDI INNOVATIVE TECHNOLOGY RESEARCH INSTITUTE COMPANY: Government/Non-Profit (9 fam.), Company (8 fam.)
   • FAIRLIE MATTHEW: Company (1 fam.), Individual (1 fam.)
 

In [7]:
# ==============================================
# COLOR SCHEME FOR SECTORS
# ==============================================

# Define color scheme for different sectors
SECTOR_COLORS = {
    'Company': '#1f77b4',                    # Blue
    'University': '#ff7f0e',                 # Orange  
    'Research Institution': '#2ca02c',       # Green
    'Government/Non-Profit': '#d62728',      # Red
    'Healthcare/Hospital': '#9467bd',        # Purple
    'Individual': '#8c564b',                 # Brown
    'Other': '#e377c2',                      # Pink
    'Unknown': '#7f7f7f'                     # Gray
}

print("🎨 Color scheme defined for sector visualization:")
for sector, color in SECTOR_COLORS.items():
    print(f"   {sector}: {color}")

🎨 Color scheme defined for sector visualization:
   Company: #1f77b4
   University: #ff7f0e
   Research Institution: #2ca02c
   Government/Non-Profit: #d62728
   Healthcare/Hospital: #9467bd
   Individual: #8c564b
   Other: #e377c2
   Unknown: #7f7f7f


In [7]:
# ==============================================
# INTERACTIVE HTML GENERATION WITH SEARCH
# ==============================================

def create_interactive_html(df, filename="electrolyzer_interactive_analysis.html"):
    """
    Create an interactive HTML file with proper search functionality and sector-based aggregation
    """
    if df.empty:
        print("⚠️ No data to create HTML file")
        return
    
    print(f"🔧 Creating interactive HTML file: {filename}")
    
    # Prepare data for JavaScript
    df_json = df.to_json(orient='records')

    # Compute multi-sector warning (same logic as notebook cell output)
    multi_sector = (
        df.groupby('Applicant_Name')['Sector']
        .nunique()
        .reset_index()
        .rename(columns={'Sector': 'n_sectors'})
    )
    multi_sector = multi_sector[multi_sector['n_sectors'] > 1].sort_values('n_sectors', ascending=False)
    if not multi_sector.empty:
        warning_rows = []
        for _, row in multi_sector.iterrows():
            name = row["Applicant_Name"]
            sectors = df.loc[df["Applicant_Name"] == name, ["Sector", "Patent_Families"]].to_dict("records")
            sector_str = " | ".join("<b>" + s["Sector"] + "</b> (" + str(s["Patent_Families"]) + " fam.)" for s in sectors)
            warning_rows.append("<li><code>" + name + "</code>: " + sector_str + "</li>")
        n = len(multi_sector)
        rows_html = "".join(warning_rows)
        q = chr(34)
        sq = chr(39)
        dismiss = ("<button onclick=" + q + "this.parentElement.style.display=" + sq + "none" + sq + q
                   + " style=" + q + "float:right;background:none;border:none;cursor:pointer;font-size:1.1em;" + q + ">&#10005;</button>")
        warning_html = (
            "<div style=" + q + "background:#fff3cd;border:1px solid #ffc107;"
            "border-radius:6px;padding:14px 18px;margin:16px 0;font-size:0.92em;" + q + ">"
            + dismiss
            + "<strong>&#9888; Data Quality Notice</strong> &mdash; "
            + str(n) + " applicant(s) appear under <em>more than one sector label</em> in PATSTAT. "
            "Their family count on each card reflects a single (Name,&nbsp;Sector) row, while the "
            "co-applicant overlap table merges all rows by name and may show a slightly higher total. "
            "This mirrors the original PATSTAT classification and is <em>not</em> corrected automatically. "
            "Please verify the sector assignment for the applicant(s) listed below before drawing conclusions."
            "<ul style=" + q + "margin:8px 0 4px 0;" + q + ">" + rows_html + "</ul>"
            "</div>"
        )
    else:
        warning_html = ""

    html_content = f"""
<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>Electrolyzer Patent Analysis - Interactive Sector Search</title>
    <script>__PLOTLY_JS_PLACEHOLDER__</script>
    <style>
        body {{
            font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif;
            margin: 20px;
            background-color: #f8f9fa;
        }}
        .container {{
            max-width: 1400px;
            margin: 0 auto;
            background-color: white;
            padding: 30px;
            border-radius: 10px;
            box-shadow: 0 4px 6px rgba(0,0,0,0.1);
        }}
        .header {{
            text-align: center;
            margin-bottom: 30px;
            padding-bottom: 20px;
            border-bottom: 3px solid #007bff;
        }}
        .search-section {{
            background-color: #f8f9fa;
            padding: 25px;
            border-radius: 8px;
            margin-bottom: 30px;
            border-left: 4px solid #28a745;
        }}
        .search-input {{
            width: 100%;
            padding: 12px;
            font-size: 16px;
            border: 2px solid #dee2e6;
            border-radius: 6px;
            margin-bottom: 15px;
            transition: border-color 0.3s;
        }}
        .search-input:focus {{
            outline: none;
            border-color: #007bff;
        }}
        .results-section {{
            display: grid;
            grid-template-columns: 1fr 1fr;
            gap: 30px;
            margin-bottom: 30px;
        }}
        .results-list {{
            max-height: 500px;
            overflow-y: auto;
            border: 1px solid #dee2e6;
            border-radius: 6px;
            background-color: white;
        }}
        .applicant-item {{
            padding: 12px;
            border-bottom: 1px solid #dee2e6;
            cursor: pointer;
            transition: background-color 0.2s;
            display: flex;
            align-items: center;
        }}
        .applicant-item:hover {{
            background-color: #f8f9fa;
        }}
        .applicant-item.selected {{
            background-color: #e3f2fd;
            border-left: 4px solid #2196f3;
        }}
        .applicant-item.hidden {{
            display: none;
        }}
        .sector-badge {{
            padding: 4px 8px;
            border-radius: 12px;
            font-size: 11px;
            font-weight: bold;
            color: white;
            margin-right: 10px;
            min-width: 80px;
            text-align: center;
        }}
        .aggregation-panel {{
            background-color: #e8f5e8;
            padding: 20px;
            border-radius: 8px;
            border-left: 4px solid #28a745;
        }}
        .total-display {{
            font-size: 24px;
            font-weight: bold;
            color: #28a745;
            text-align: center;
            margin: 15px 0;
        }}
        .btn {{
            padding: 10px 20px;
            border: none;
            border-radius: 6px;
            cursor: pointer;
            font-size: 14px;
            margin: 5px;
            transition: all 0.3s;
        }}
        .btn-primary {{ background-color: #007bff; color: white; }}
        .btn-success {{ background-color: #28a745; color: white; }}
        .btn-danger {{ background-color: #dc3545; color: white; }}
        .btn:hover {{ transform: translateY(-2px); box-shadow: 0 4px 8px rgba(0,0,0,0.2); }}
        .stats-grid {{
            display: grid;
            grid-template-columns: repeat(auto-fit, minmax(200px, 1fr));
            gap: 15px;
            margin: 20px 0;
        }}
        .stat-card {{
            background-color: white;
            padding: 15px;
            border-radius: 6px;
            border-left: 4px solid #007bff;
            text-align: center;
        }}
        .visualization-section {{
            margin-top: 30px;
            padding: 20px;
            background-color: #f8f9fa;
            border-radius: 8px;
        }}
        .info-box {{
            background-color: #d1ecf1;
            border: 1px solid #bee5eb;
            border-radius: 6px;
            padding: 15px;
            margin-bottom: 20px;
        }}
        .match-count {{
            background-color: #fff3cd;
            border: 1px solid #ffeaa7;
            border-radius: 6px;
            padding: 10px;
            margin-bottom: 10px;
            text-align: center;
            font-weight: bold;
        }}
        .overcount-warning {{
            background-color: #fff8e1;
            border: 1px solid #ffc107;
            border-left: 4px solid #e65100;
            border-radius: 6px;
            padding: 10px 14px;
            margin-top: 12px;
            font-size: 13px;
            color: #5d4000;
            display: none;
        }}
        .unique-display {{
            font-size: 20px;
            font-weight: bold;
            color: #0277bd;
            text-align: center;
            margin: 8px 0 4px 0;
            display: none;
        }}
        .overlap-detail {{
            background-color: #fce4ec;
            border: 1px solid #e91e63;
            border-left: 4px solid #c62828;
            border-radius: 6px;
            padding: 10px 14px;
            margin-top: 8px;
            font-size: 12px;
            color: #3e0000;
            display: none;
        }}
        .no-overlap-note {{
            background-color: #e8f5e9;
            border: 1px solid #81c784;
            border-left: 4px solid #2e7d32;
            border-radius: 6px;
            padding: 8px 12px;
            margin-top: 8px;
            font-size: 12px;
            color: #1b5e20;
            display: none;
        }}
    </style>
</head>
<body>
    <div class="container">
        <div class="header">
            <h1>🔬 Electrolyzer Patent Analysis</h1>
            <h2>Interactive Sector-Based Applicant Search & Aggregation</h2>
            <p>Search and aggregate patent families by applicant names and sectors</p>
        </div>
        {warning_html}
        <div class="search-section">
            <h3>🔍 Smart Applicant Search</h3>
            <input type="text" id="searchInput" class="search-input" 
                   placeholder="Enter keyword to filter applicants (e.g., 'TOSHIBA', 'UNIVERSITY', 'HUANENG')..."
                   oninput="filterApplicants()">
            <div class="info-box">
                <strong>Instructions:</strong> 
                <ul style="margin: 10px 0;">
                    <li>Type a keyword to filter the complete list of applicants below</li>
                    <li>Click on applicant names to select/deselect them</li>
                    <li>See real-time aggregated patent family totals for selected applicants</li>
                    <li>Use "Select All Visible" to select all currently filtered applicants</li>
                </ul>
            </div>
            <div class="match-count" id="matchCount">Showing all {len(df)} applicants</div>
        </div>
        
        <div class="results-section">
            <div>
                <h3>📋 Applicant List</h3>
                <div class="btn-group" style="margin-bottom: 10px;">
                    <button class="btn btn-success" onclick="selectAllVisible()">Select All Visible</button>
                    <button class="btn btn-danger" onclick="deselectAll()">Deselect All</button>
                    <button class="btn btn-primary" onclick="clearSearch()">Clear Search</button>
                </div>
                <div id="results" class="results-list">
                    <!-- Pre-loaded applicants will be here -->
                </div>
            </div>
            
            <div>
                <h3>📊 Aggregation Results</h3>
                <div class="aggregation-panel">
                    <h4>Selected Applicants Summary</h4>
                    <div class="total-display" id="totalPatents">0 Patent Families</div>
                    <div class="overcount-warning" id="overestimationWarning">
                        &#9888; <strong>Caution &mdash; possible overestimation.</strong>
                        When multiple applicants are selected, their patent family counts are simply added.
                        If any of the selected applicants are <em>co-applicants on the same patent families</em>
                        (e.g. different name variants of the same organisation, or partners listed separately
                        on a joint filing), those families will be counted more than once and the total
                        above will be inflated accordingly.
                        This risk is particularly relevant when grouping applicants that share a common
                        keyword (e.g. "HUANENG") but whose relationship &mdash; distinct entities vs.
                        co-owners of the same portfolio &mdash; is uncertain.
                    </div>
                    <div class="unique-display" id="uniquePatents"></div>
                    <div class="overlap-detail" id="overlapDetail"></div>
                    <div class="no-overlap-note" id="noOverlapNote">&#10003; No shared patent families detected among the selected applicants.</div>
                    <div class="stats-grid">
                        <div class="stat-card">
                            <h4 id="selectedCount">0</h4>
                            <p>Selected Applicants</p>
                        </div>
                        <div class="stat-card">
                            <h4 id="sectorCount">0</h4>
                            <p>Different Sectors</p>
                        </div>
                        <div class="stat-card">
                            <h4 id="visibleCount">{len(df)}</h4>
                            <p>Visible Applicants</p>
                        </div>
                    </div>
                    <div id="sectorBreakdown"></div>
                </div>
            </div>
        </div>
        
        <div class="visualization-section">
            <h3>📈 Visualization</h3>
            <div id="chartContainer" style="height: 500px;"></div>
        </div>
        
        <div style="margin-top: 30px; text-align: center;">
            <button class="btn btn-primary" onclick="exportSelected()">📥 Export Selected Results (CSV)</button>
            <button class="btn btn-primary" onclick="exportAll()">📊 Download All Data (CSV)</button>
        </div>
    </div>

    <script>
        // Data from Python
        const allApplicants = {df_json};
        
        // Sector colors
        const sectorColors = {{
            'Company': '#1f77b4',
            'University': '#ff7f0e',
            'Research Institution': '#2ca02c',
            'Government/Non-Profit': '#d62728',
            'Healthcare/Hospital': '#9467bd',
            'Individual': '#8c564b',
            'Other': '#e377c2',
            'Unknown': '#7f7f7f'
        }};
        
        let selectedApplicants = new Set();
        
        // Initialize the page
        document.addEventListener('DOMContentLoaded', function() {{
            loadAllApplicants();
            updateAggregation();
        }});
        
        function loadAllApplicants() {{
            const resultsDiv = document.getElementById('results');
            // Escape HTML special characters to prevent broken markup
            function escHtml(s) {{ return String(s).replace(/&/g,'&amp;').replace(/</g,'&lt;').replace(/>/g,'&gt;').replace(/"/g,'&quot;').replace(/'/g,'&#39;'); }}
            resultsDiv.innerHTML = allApplicants.map(item => {{
                const safeName = escHtml(item.Applicant_Name);
                return `<div class="applicant-item" onclick="toggleSelection(this.dataset.fullname)" 
                     id="applicant-${{item.Rank}}" data-name="${{escHtml(item.Applicant_Name).toLowerCase()}}"
                     data-fullname="${{safeName}}">
                    <span class="sector-badge" style="background-color: ${{sectorColors[item.Sector] || '#7f7f7f'}}">
                        ${{item.Sector}}
                    </span>
                    <div>
                        <strong>${{safeName}}</strong><br>
                        <small>${{item.Patent_Families}} families (${{item.First_Year}}-${{item.Last_Year}})</small>
                    </div>
                </div>`;
            }}).join('');
        }}
        
        function filterApplicants() {{
            const query = document.getElementById('searchInput').value.toLowerCase().trim();
            let visibleCount = 0;
            
            allApplicants.forEach(item => {{
                const element = document.getElementById(`applicant-${{item.Rank}}`);
                if (element) {{
                    if (query === '' || item.Applicant_Name.toLowerCase().includes(query)) {{
                        element.classList.remove('hidden');
                        visibleCount++;
                    }} else {{
                        element.classList.add('hidden');
                    }}
                }}
            }});
            
            // Update match count
            const matchCountDiv = document.getElementById('matchCount');
            if (query === '') {{
                matchCountDiv.textContent = `Showing all ${{allApplicants.length}} applicants`;
            }} else {{
                matchCountDiv.textContent = `Showing ${{visibleCount}} applicants matching "${{query}}"`;
            }}
            
            // Update visible count
            document.getElementById('visibleCount').textContent = visibleCount;
        }}
        
        function toggleSelection(applicantName) {{
            // Decode HTML entities from data-attribute
            const tmp = document.createElement('textarea');
            tmp.innerHTML = applicantName;
            const decoded = tmp.value;
            if (selectedApplicants.has(decoded)) {{
                selectedApplicants.delete(decoded);
            }} else {{
                selectedApplicants.add(decoded);
            }}
            updateSelectionDisplay();
            updateAggregation();
        }}
        
        function selectAllVisible() {{
            allApplicants.forEach(item => {{
                const element = document.getElementById(`applicant-${{item.Rank}}`);
                if (element && !element.classList.contains('hidden')) {{
                    selectedApplicants.add(item.Applicant_Name);
                }}
            }});
            updateSelectionDisplay();
            updateAggregation();
        }}
        
        function deselectAll() {{
            selectedApplicants.clear();
            updateSelectionDisplay();
            updateAggregation();
        }}
        
        function clearSearch() {{
            document.getElementById('searchInput').value = '';
            filterApplicants();
        }}
        
        function updateSelectionDisplay() {{
            allApplicants.forEach(item => {{
                const element = document.getElementById(`applicant-${{item.Rank}}`);
                if (element) {{
                    if (selectedApplicants.has(item.Applicant_Name)) {{
                        element.classList.add('selected');
                    }} else {{
                        element.classList.remove('selected');
                    }}
                }}
            }});
        }}
        
        function updateAggregation() {{
            const selectedData = allApplicants.filter(item => 
                selectedApplicants.has(item.Applicant_Name)
            );
            
            const totalPatents = selectedData.reduce((sum, item) => sum + item.Patent_Families, 0);
            const uniqueSectors = [...new Set(selectedData.map(item => item.Sector))];
            
            document.getElementById('totalPatents').textContent = `${{totalPatents.toLocaleString()}} Patent Families`;
            document.getElementById('selectedCount').textContent = selectedData.length;
            document.getElementById('sectorCount').textContent = uniqueSectors.length;

            // ── Deduplication & multi-way overlap analysis ────────────────────────────
            // Step 1: merge Family_ID_List per Applicant_Name.
            // The same entity can appear with different sector labels in PATSTAT,
            // producing multiple rows for the same name → merge their id sets first.
            const dedupMap = new Map();
            selectedData.forEach(item => {{
                if (!dedupMap.has(item.Applicant_Name)) dedupMap.set(item.Applicant_Name, new Set());
                (item.Family_ID_List || []).forEach(id => dedupMap.get(item.Applicant_Name).add(id));
            }});
            const actors  = [...dedupMap.entries()].map(([name, ids]) => ({{ name, ids, total: ids.size }}));
            const nActors = actors.length;
            const hasIdData = actors.some(a => a.ids.size > 0);

            // Step 2: global union + per-family frequency map
            const globalUnion = new Set();
            const freqMap = new Map();  // familyId -> how many distinct actors own it
            actors.forEach(a => {{
                a.ids.forEach(id => {{
                    globalUnion.add(id);
                    freqMap.set(id, (freqMap.get(id) || 0) + 1);
                }});
            }});
            const uniqueTotal  = globalUnion.size;
            const dedupSum     = actors.reduce((s, a) => s + a.total, 0);
            const overlapCount = dedupSum - uniqueTotal;

            // Step 3: per-actor exclusive (not shared) vs shared (with >=1 other)
            actors.forEach(a => {{
                a.exclusive = 0;  a.shared = 0;
                a.ids.forEach(id => {{ if (freqMap.get(id) === 1) a.exclusive++; else a.shared++; }});
            }});

            // Step 4: sharing-level distribution
            // levelDist[k] = number of families owned by exactly k of the selected actors
            const levelDist = new Map();
            freqMap.forEach(cnt => levelDist.set(cnt, (levelDist.get(cnt) || 0) + 1));

            // Step 5: pairwise overlaps – canonical key prevents A↔B / B↔A duplicates
            const pairMap2 = new Map();
            for (let ii = 0; ii < actors.length; ii++) {{
                for (let jj = ii + 1; jj < actors.length; jj++) {{
                    const cnt2 = [...actors[jj].ids].filter(id => actors[ii].ids.has(id)).length;
                    if (cnt2 > 0) {{
                        const pkey = [actors[ii].name, actors[jj].name].sort().join('||||');
                        if (!pairMap2.has(pkey) || cnt2 > pairMap2.get(pkey).count) {{
                            pairMap2.set(pkey, {{ a: actors[ii].name, b: actors[jj].name, count: cnt2 }});
                        }}
                    }}
                }}
            }}
            const pairList = [...pairMap2.values()].sort((x, y) => y.count - x.count);

            // ── Update display elements ──────────────────────────────────────────────
            const uniqueElem  = document.getElementById('uniquePatents');
            const overlapElem = document.getElementById('overlapDetail');
            const noOverlapEl = document.getElementById('noOverlapNote');
            const warnElem    = document.getElementById('overestimationWarning');

            if (nActors > 1 && hasIdData) {{
                document.getElementById('totalPatents').innerHTML =
                    totalPatents.toLocaleString() + ' Patent Families'
                    + '<br><small style="font-size:12px;color:#555;font-weight:normal;">(naive sum)</small>';

                uniqueElem.textContent = uniqueTotal.toLocaleString() + ' Unique Patent Families';
                uniqueElem.style.display = 'block';

                if (overlapCount > 0) {{
                    warnElem.innerHTML =
                        '&#9888;&nbsp;<strong>Overlap detected.</strong>'
                        + ' Naive sum overcounts by&nbsp;<strong>'
                        + overlapCount.toLocaleString() + '</strong>&nbsp;patent&nbsp;'
                        + (overlapCount === 1 ? 'family' : 'families')
                        + '&nbsp;(co-applicant families counted more than once). See breakdown below.';
                    warnElem.style.display = 'block';
                    noOverlapEl.style.display = 'none';

                    // ─ Per-actor breakdown table ─────────────────────────────────────
                    const rowsHtml = actors.map(a =>
                        '<tr>'
                        + '<td style="padding:4px 8px;word-break:break-word;">' + a.name + '</td>'
                        + '<td style="padding:4px 8px;text-align:center;">' + a.total.toLocaleString() + '</td>'
                        + '<td style="padding:4px 8px;text-align:center;color:#1b5e20;">' + a.exclusive.toLocaleString() + '</td>'
                        + '<td style="padding:4px 8px;text-align:center;color:#c62828;">' + a.shared.toLocaleString() + '</td>'
                        + '</tr>'
                    ).join('');
                    const tableHtml =
                        '<table style="width:100%;border-collapse:collapse;margin-bottom:8px;font-size:12px;">'
                        + '<thead><tr style="background:#f8bbd9;font-weight:bold;">'
                        + '<th style="padding:4px 8px;text-align:left;font-weight:bold;">Applicant</th>'
                        + '<th style="padding:4px 8px;" title="Total unique families for this applicant">Total</th>'
                        + '<th style="padding:4px 8px;color:#1b5e20;" title="Families not shared with any other selected applicant">Exclusive</th>'
                        + '<th style="padding:4px 8px;color:#c62828;" title="Families shared with at least one other selected applicant">Shared</th>'
                        + '</tr></thead><tbody>' + rowsHtml + '</tbody></table>';

                    // ─ Sharing-level distribution ────────────────────────────────────
                    const levels = [...levelDist.entries()].sort((a, b) => a[0] - b[0]);
                    const levelHtml = '<div style="margin-bottom:8px;">'
                        + '<strong>Sharing-level breakdown:</strong>'
                        + levels.map(([k, cnt]) => {{
                            const lbl = k === 1
                                ? (nActors === 2
                                    ? 'exclusive to exactly one of the 2 applicants'
                                    : 'exclusive to exactly 1 of the ' + nActors + ' selected applicants')
                                : (k === nActors
                                    ? 'shared by <strong>all&nbsp;' + nActors + '</strong> selected applicants'
                                    : 'shared by exactly ' + k + ' of the ' + nActors + ' selected applicants');
                            return '<div style="padding:2px 0 2px 8px;">&bull;&nbsp;<strong>'
                                + cnt.toLocaleString() + '</strong>&nbsp;'
                                + (cnt === 1 ? 'family' : 'families') + '&nbsp;' + lbl + '</div>';
                        }}).join('')
                        + '</div>';

                    // ─ Pairwise detail ───────────────────────────────────────────────
                    let pairsHtml = '';
                    if (actors.length <= 100) {{
                        if (pairList.length > 0) {{
                            const shown = pairList.slice(0, 30);
                            pairsHtml =
                                '<div style="margin-top:2px;"><strong>Pairwise overlaps:</strong>'
                                + shown.map(p =>
                                    '<div style="padding:2px 0 2px 8px;border-bottom:1px solid #f8bbd9;">'
                                    + '<strong>' + p.count.toLocaleString() + '</strong>&nbsp;shared&nbsp;'
                                    + (p.count === 1 ? 'family' : 'families') + ':&nbsp;'
                                    + '<em>' + p.a + '</em>&nbsp;&harr;&nbsp;<em>' + p.b + '</em></div>'
                                ).join('')
                                + (pairList.length > 30
                                    ? '<div style="color:#888;">&hellip;and&nbsp;'
                                      + (pairList.length - 30) + '&nbsp;more pairs</div>'
                                    : '')
                                + '</div>';
                        }}
                    }} else {{
                        pairsHtml = '<div style="color:#888;margin-top:4px;">'
                            + '(Pairwise detail hidden for &gt;100 selected applicants)</div>';
                    }}

                    overlapElem.innerHTML =
                        '<strong>Co-applicant overlap breakdown</strong>'
                        + '<div style="margin-top:8px;">' + tableHtml + '</div>'
                        + levelHtml
                        + pairsHtml;
                    overlapElem.style.display = 'block';

                }} else {{
                    warnElem.style.display    = 'none';
                    overlapElem.style.display = 'none';
                    noOverlapEl.style.display = 'block';
                }}

            }} else {{
                document.getElementById('totalPatents').textContent =
                    totalPatents.toLocaleString() + ' Patent Families';
                uniqueElem.style.display  = 'none';
                overlapElem.style.display = 'none';
                noOverlapEl.style.display = 'none';
                warnElem.style.display    = 'none';
            }}

            // Sector breakdown
            const sectorBreakdown = {{}};
            selectedData.forEach(item => {{
                if (!sectorBreakdown[item.Sector]) {{
                    sectorBreakdown[item.Sector] = {{ count: 0, patents: 0 }};
                }}
                sectorBreakdown[item.Sector].count++;
                sectorBreakdown[item.Sector].patents += item.Patent_Families;
            }});
            
            const breakdownHtml = Object.entries(sectorBreakdown).map(([sector, data]) => `
                <div style="display: flex; justify-content: space-between; padding: 5px 0; border-bottom: 1px solid #ddd;">
                    <span style="color: ${{sectorColors[sector]}}; font-weight: bold;">${{sector}}</span>
                    <span>${{data.count}} applicants, ${{data.patents}} families</span>
                </div>
            `).join('');
            
            document.getElementById('sectorBreakdown').innerHTML = breakdownHtml;
            
            // Update visualization
            updateVisualization(selectedData);
        }}
        
        function updateVisualization(data) {{
            if (data.length === 0) {{
                document.getElementById('chartContainer').innerHTML = '<p style="text-align: center; padding: 50px; color: #666;">Select applicants to see visualization</p>';
                return;
            }}
            
            // Create sector-based pie chart
            const sectorData = {{}};
            data.forEach(item => {{
                if (!sectorData[item.Sector]) {{
                    sectorData[item.Sector] = 0;
                }}
                sectorData[item.Sector] += item.Patent_Families;
            }});
            
            const labels = Object.keys(sectorData);
            const values = Object.values(sectorData);
            const colors = labels.map(label => sectorColors[label] || '#7f7f7f');
            
            const trace = {{
                type: 'pie',
                labels: labels,
                values: values,
                marker: {{ colors: colors }},
                textinfo: 'label+percent+value',
                hovertemplate: '<b>%{{label}}</b><br>Patent Families: %{{value}}<br>Percentage: %{{percent}}<extra></extra>'
            }};
            
            const layout = {{
                title: 'Patent Families by Sector (Selected Applicants)',
                font: {{ size: 14 }},
                showlegend: true
            }};
            
            Plotly.newPlot('chartContainer', [trace], layout);
        }}
        
        function exportSelected() {{
            const selectedData = allApplicants.filter(item => 
                selectedApplicants.has(item.Applicant_Name)
            );
            
            if (selectedData.length === 0) {{
                alert('Please select applicants first');
                return;
            }}
            
            const csv = convertToCSV(selectedData);
            downloadCSV(csv, 'selected_electrolyzer_applicants.csv');
        }}
        
        function exportAll() {{
            const csv = convertToCSV(allApplicants);
            downloadCSV(csv, 'all_electrolyzer_applicants.csv');
        }}
        
        function convertToCSV(data) {{
            const headers = ['Rank', 'Applicant_Name', 'Sector', 'Patent_Families', 'First_Year', 'Last_Year'];
            const csvContent = [headers.join(',')];
            
            data.forEach(row => {{
                const values = headers.map(header => {{
                    let value = row[header] || '';
                    if (typeof value === 'string' && value.includes(',')) {{
                        value = `"${{value}}"`;
                    }}
                    return value;
                }});
                csvContent.push(values.join(','));
            }});
            
            return csvContent.join('\\n');
        }}
        
        function downloadCSV(csv, filename) {{
            const blob = new Blob([csv], {{ type: 'text/csv' }});
            const url = window.URL.createObjectURL(blob);
            const a = document.createElement('a');
            a.href = url;
            a.download = filename;
            a.click();
            window.URL.revokeObjectURL(url);
        }}
    </script>
</body>
</html>
    """

    # Embed plotly.js inline (no CDN dependency)
    import plotly as _pl
    import os as _os
    with open(_os.path.join(_os.path.dirname(_pl.__file__), "package_data", "plotly.min.js")) as _f:
        html_content = html_content.replace("__PLOTLY_JS_PLACEHOLDER__", _f.read())
    
    # Write HTML file
    with open(filename, 'w', encoding='utf-8') as f:
        f.write(html_content)
    
    print(f"✅ Interactive HTML file created: {filename}")
    print(f"🌐 Open {filename} in your browser to use the interactive search and aggregation features")
    print(f"🔍 Features:")
    print(f"   • Pre-loaded list of all {len(df)} applicants")
    print(f"   • Real-time search filtering (e.g., type 'TOSHIBA' to see only TOSHIBA-related names)")
    print(f"   • Click to select/deselect individual applicants")
    print(f"   • Live aggregation of patent families for selected applicants")
    print(f"   • Color-coded sector visualization")

# Create the interactive HTML file
if not applicant_summary.empty:
    create_interactive_html(applicant_summary)
else:
    print("⚠️ Cannot create HTML file - no data available")

🔧 Creating interactive HTML file: electrolyzer_interactive_analysis.html
✅ Interactive HTML file created: electrolyzer_interactive_analysis.html
🌐 Open electrolyzer_interactive_analysis.html in your browser to use the interactive search and aggregation features
🔍 Features:
   • Pre-loaded list of all 12849 applicants
   • Real-time search filtering (e.g., type 'TOSHIBA' to see only TOSHIBA-related names)
   • Click to select/deselect individual applicants
   • Live aggregation of patent families for selected applicants
   • Color-coded sector visualization


In [8]:
# ==============================================
# ADVANCED VISUALIZATIONS WITH PLOTLY
# ==============================================

import base64
from io import BytesIO

print("📊 Creating advanced Plotly visualizations...")

if not applicant_summary.empty:
    # Create sector distribution data for visualization
    sector_data = applicant_summary.groupby('Sector')['Patent_Families'].sum().reset_index()
    sector_data['Percentage'] = (sector_data['Patent_Families'] / sector_data['Patent_Families'].sum()) * 100
    sector_data = sector_data.sort_values('Patent_Families', ascending=False)
    
    # Create sector distribution bar chart
    fig_bar = go.Figure()
    fig_bar.add_trace(go.Bar(
        x=sector_data['Patent_Families'],
        y=sector_data['Sector'],
        orientation='h',
        marker_color=[SECTOR_COLORS.get(sector, '#7f7f7f') for sector in sector_data['Sector']],
        text=sector_data['Patent_Families'],
        textposition='auto',
        hovertemplate='<b>%{y}</b><br>Patent Families: %{x}<br>Percentage: %{customdata:.1f}%<extra></extra>',
        customdata=sector_data['Percentage']
    ))

    fig_bar.update_layout(
        title={
            'text': 'Electrolyzer Patent Families by Applicant Sector',
            'x': 0.5,
            'xanchor': 'center',
            'font': {'size': 16, 'family': 'Arial Black'}
        },
        xaxis_title='Number of Patent Families',
        yaxis_title='Applicant Sector',
        height=600,
        margin=dict(l=200, r=50, t=80, b=50),
        showlegend=False,
        plot_bgcolor='white',
        paper_bgcolor='white'
    )

    fig_bar.update_xaxes(gridcolor='lightgray')
    fig_bar.update_yaxes(gridcolor='lightgray')

    print("✅ Bar chart created")

    # Create sector distribution pie chart
    fig_pie = go.Figure()
    fig_pie.add_trace(go.Pie(
        labels=sector_data['Sector'],
        values=sector_data['Patent_Families'],
        hole=0.4,
        marker=dict(colors=[SECTOR_COLORS.get(sector, '#7f7f7f') for sector in sector_data['Sector']]),
        textinfo='label+percent',
        textposition='outside',
        hovertemplate='<b>%{label}</b><br>Patent Families: %{value}<br>Percentage: %{percent}<extra></extra>'
    ))

    fig_pie.update_layout(
        title={
            'text': 'Sector Distribution of Electrolyzer Patents',
            'x': 0.5,
            'xanchor': 'center',
            'font': {'size': 16, 'family': 'Arial Black'}
        },
        height=600,
        showlegend=True,
        legend=dict(
            orientation="v",
            yanchor="middle",
            y=0.5,
            xanchor="left",
            x=1.05
        ),
        margin=dict(l=50, r=150, t=80, b=50)
    )

    print("✅ Pie chart created")

    # Create top applicants by sector scatter plot
    top_applicants_by_sector = applicant_summary.head(30)  # Top 30 for visualization

    fig_scatter = go.Figure()

    for sector in top_applicants_by_sector['Sector'].unique():
        sector_data_scatter = top_applicants_by_sector[top_applicants_by_sector['Sector'] == sector]
        
        fig_scatter.add_trace(go.Scatter(
            x=sector_data_scatter['Patent_Families'],
            y=sector_data_scatter['Applicant_Name'],
            mode='markers',
            marker=dict(
                color=SECTOR_COLORS.get(sector, '#7f7f7f'),
                size=12,
                line=dict(width=1, color='white')
            ),
            name=sector,
            text=sector_data_scatter['Applicant_Name'],
            hovertemplate='<b>%{text}</b><br>Sector: ' + sector + '<br>Patent Families: %{x}<extra></extra>'
        ))

    fig_scatter.update_layout(
        title={
            'text': 'Top 30 Electrolyzer Patent Applicants by Sector',
            'x': 0.5,
            'xanchor': 'center',
            'font': {'size': 16, 'family': 'Arial Black'}
        },
        xaxis_title='Number of Patent Families',
        yaxis_title='Applicant',
        height=800,
        margin=dict(l=300, r=50, t=80, b=50),
        showlegend=True,
        legend=dict(
            orientation="v",
            yanchor="top",
            y=1,
            xanchor="left",
            x=1.02
        ),
        plot_bgcolor='white',
        paper_bgcolor='white'
    )

    fig_scatter.update_xaxes(gridcolor='lightgray')
    fig_scatter.update_yaxes(gridcolor='lightgray')

    print("✅ Scatter plot created")

    # Display all charts
    fig_bar.show()
    fig_pie.show()
    fig_scatter.show()

    print("📊 All advanced visualizations displayed!")

    # Store figures for download functionality
    viz_figures = {
        'bar_chart': fig_bar,
        'pie_chart': fig_pie,
        'scatter_plot': fig_scatter
    }
    
else:
    print("⚠️ No data available for visualizations")
    viz_figures = {}

📊 Creating advanced Plotly visualizations...


NameError: name 'SECTOR_COLORS' is not defined

In [10]:
# ==============================================
# ENHANCED DOWNLOADS WITH EMBEDDED DIAGRAMS
# ==============================================

def create_enhanced_downloads(df, figures_dict):
    """
    Create comprehensive downloadable files with embedded visualizations
    """
    if df.empty:
        print("⚠️ No data available for enhanced downloads")
        return
    
    print("📁 Creating enhanced downloadable files...")
    
    # 1. Create comprehensive Excel file with multiple sheets and embedded charts
    excel_filename = "Electrolyzer_Patent_Analysis_Enhanced_Complete.xlsx"
    
    with pd.ExcelWriter(excel_filename, engine='openpyxl') as writer:
        # Main summary sheet
        df.to_excel(writer, sheet_name='Applicant_Summary', index=False)
        
        # Sector analysis sheet
        sector_summary = df.groupby('Sector').agg({
            'Patent_Families': ['count', 'sum', 'mean', 'median', 'max'],
            'First_Year': 'min',
            'Last_Year': 'max'
        }).reset_index()
        
        # Flatten column names
        sector_summary.columns = ['Sector', 'Applicant_Count', 'Total_Families', 'Avg_Families', 'Median_Families', 'Max_Families', 'Earliest_Year', 'Latest_Year']
        sector_summary = sector_summary.sort_values('Total_Families', ascending=False)
        sector_summary.to_excel(writer, sheet_name='Sector_Analysis', index=False)
        
        # Top applicants by sector
        for sector in df['Sector'].unique():
            sector_data = df[df['Sector'] == sector].head(20)
            sheet_name = f"Top_{sector.replace('/', '_').replace(' ', '_')}"[:31]  # Excel sheet name limit
            sector_data.to_excel(writer, sheet_name=sheet_name, index=False)
        
        # Statistics sheet
        stats_data = {
            'Metric': [
                'Total Applicants',
                'Total Patent Families',
                'Average Families per Applicant',
                'Median Families per Applicant',
                'Top Applicant Patent Families',
                'Number of Sectors',
                'Year Range',
                'Analysis Date'
            ],
            'Value': [
                len(df),
                df['Patent_Families'].sum(),
                f"{df['Patent_Families'].mean():.2f}",
                f"{df['Patent_Families'].median():.0f}",
                df['Patent_Families'].max(),
                df['Sector'].nunique(),
                f"{df['First_Year'].min()}-{df['Last_Year'].max()}",
                datetime.now().strftime('%Y-%m-%d %H:%M:%S')
            ]
        }
        
        stats_df = pd.DataFrame(stats_data)
        stats_df.to_excel(writer, sheet_name='Statistics', index=False)
    
    print(f"✅ Enhanced Excel file created: {excel_filename}")
    
    # 2. Create comprehensive HTML report with embedded charts
    html_filename = "Electrolyzer_Patent_Analysis_Enhanced_Report.html"
    
    # Convert Plotly figures to HTML
    chart_htmls = {}
    if figures_dict:
        for chart_name, fig in figures_dict.items():
            chart_htmls[chart_name] = fig.to_html(include_plotlyjs=False, full_html=False, div_id=f"chart_{chart_name}")
    
    # Create sector distribution table HTML
    sector_summary_html = sector_summary.to_html(classes='table table-striped', index=False)
    
    # Create top applicants table HTML
    top_applicants_html = df.head(50).to_html(classes='table table-striped', index=False)
    
    html_content = f"""
<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>Electrolyzer Patent Analysis - Enhanced Report</title>
    <link href="https://cdn.jsdelivr.net/npm/bootstrap@5.1.3/dist/css/bootstrap.min.css" rel="stylesheet">
    <script>__PLOTLY_JS_PLACEHOLDER__</script>
    <style>
        body {{ font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif; }}
        .header {{ background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); color: white; padding: 60px 0; text-align: center; }}
        .section {{ margin: 40px 0; }}
        .chart-container {{ margin: 30px 0; padding: 20px; border: 1px solid #dee2e6; border-radius: 10px; }}
        .stat-card {{ background: #f8f9fa; padding: 20px; border-radius: 10px; text-align: center; margin: 10px 0; }}
        .table {{ font-size: 0.9em; }}
        .sector-badge {{ padding: 4px 8px; border-radius: 12px; font-size: 11px; font-weight: bold; color: white; }}
    </style>
</head>
<body>
    <div class="header">
        <div class="container">
            <h1 class="display-4">🔬 Electrolyzer Patent Analysis</h1>
            <h2>Enhanced Sector-Based Analysis Report</h2>
            <p class="lead">Comprehensive analysis of electrolyzer patent applicants with sector classification</p>
            <p>Generated on: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}</p>
        </div>
    </div>
    
    <div class="container">
        <div class="section">
            <h2>📊 Executive Summary</h2>
            <div class="row">
                <div class="col-md-3">
                    <div class="stat-card">
                        <h3>{len(df):,}</h3>
                        <p>Total Applicants</p>
                    </div>
                </div>
                <div class="col-md-3">
                    <div class="stat-card">
                        <h3>{df['Patent_Families'].sum():,}</h3>
                        <p>Patent Families</p>
                    </div>
                </div>
                <div class="col-md-3">
                    <div class="stat-card">
                        <h3>{df['Sector'].nunique()}</h3>
                        <p>Sectors</p>
                    </div>
                </div>
                <div class="col-md-3">
                    <div class="stat-card">
                        <h3>{df['First_Year'].min()}-{df['Last_Year'].max()}</h3>
                        <p>Year Range</p>
                    </div>
                </div>
            </div>
        </div>
        
        <div class="section">
            <h2>📈 Sector Distribution Visualization</h2>
            <div class="chart-container">
                {chart_htmls.get('pie_chart', '<p>Pie chart not available</p>')}
            </div>
        </div>
        
        <div class="section">
            <h2>📊 Patent Families by Sector</h2>
            <div class="chart-container">
                {chart_htmls.get('bar_chart', '<p>Bar chart not available</p>')}
            </div>
        </div>
        
        <div class="section">
            <h2>🏆 Top Applicants Analysis</h2>
            <div class="chart-container">
                {chart_htmls.get('scatter_plot', '<p>Scatter plot not available</p>')}
            </div>
        </div>
        
        <div class="section">
            <h2>📋 Sector Analysis Summary</h2>
            <div class="table-responsive">
                {sector_summary_html}
            </div>
        </div>
        
        <div class="section">
            <h2>🔝 Top 50 Applicants</h2>
            <div class="table-responsive">
                {top_applicants_html}
            </div>
        </div>
        
        <div class="section">
            <h2>📁 Data Sources & Methodology</h2>
            <div class="row">
                <div class="col-md-6">
                    <h4>Data Sources</h4>
                    <ul>
                        <li>Electrolyzer Enhanced Final Dataset 2025</li>
                        <li>PATSTAT TLS206_PERSON table (PSN_SECTOR)</li>
                        <li>PATSTAT TLS201_APPLN table</li>
                        <li>PATSTAT TLS207_PERS_APPLN table</li>
                    </ul>
                </div>
                <div class="col-md-6">
                    <h4>Classification Methodology</h4>
                    <ul>
                        <li>Priority: Database PSN_SECTOR classification</li>
                        <li>Fallback: Enhanced name-based classification</li>
                        <li>Special handling for vague PSN_SECTOR values</li>
                        <li>Enhanced Academy of Sciences detection</li>
                    </ul>
                </div>
            </div>
        </div>
        
        <div class="section">
            <h2>🎨 Sector Color Legend</h2>
            <div class="row">
                {''.join([f'<div class="col-md-3 mb-2"><span class="sector-badge" style="background-color: {color};">{sector}</span></div>' for sector, color in SECTOR_COLORS.items()])}
            </div>
        </div>
    </div>
    
    <footer class="bg-dark text-light text-center py-4 mt-5">
        <div class="container">
            <p>© 2025 Electrolyzer Patent Analysis - Enhanced Sector Analysis</p>
            <p>Generated with enhanced sector classification and interactive features</p>
        </div>
    </footer>
</body>
</html>
    """

    # Embed plotly.js inline (no CDN dependency)
    import plotly as _pl
    import os as _os
    with open(_os.path.join(_os.path.dirname(_pl.__file__), "package_data", "plotly.min.js")) as _f:
        html_content = html_content.replace("__PLOTLY_JS_PLACEHOLDER__", _f.read())
    
    with open(html_filename, 'w', encoding='utf-8') as f:
        f.write(html_content)
    
    print(f"✅ Enhanced HTML report created: {html_filename}")
    
    # 3. Create CSV files for raw data
    csv_filename = "Electrolyzer_Patent_Analysis_Enhanced_Data.csv"
    df.to_csv(csv_filename, index=False)
    print(f"✅ CSV data file created: {csv_filename}")
    
    # 4. Create sector summary CSV
    sector_csv_filename = "Electrolyzer_Sector_Analysis_Summary.csv"
    sector_summary.to_csv(sector_csv_filename, index=False)
    print(f"✅ Sector summary CSV created: {sector_csv_filename}")
    
    print("\n🎯 ENHANCED DOWNLOAD FILES SUMMARY:")
    print("=" * 60)
    print(f"📊 Excel Report (Multi-sheet): {excel_filename}")
    print(f"🌐 HTML Report (With charts): {html_filename}")
    print(f"📋 CSV Data File: {csv_filename}")
    print(f"📈 Sector Summary CSV: {sector_csv_filename}")
    print("=" * 60)
    print("✨ All files include embedded visualizations and comprehensive analysis")
    print("🔗 The HTML file includes interactive Plotly charts")
    print("📊 The Excel file contains multiple analysis sheets")

# Create enhanced downloadable files
if not applicant_summary.empty and viz_figures:
    create_enhanced_downloads(applicant_summary, viz_figures)
else:
    print("⚠️ Cannot create enhanced downloads - missing data or visualizations")

📁 Creating enhanced downloadable files...
✅ Enhanced Excel file created: Electrolyzer_Patent_Analysis_Enhanced_Complete.xlsx
✅ Enhanced HTML report created: Electrolyzer_Patent_Analysis_Enhanced_Report.html
✅ CSV data file created: Electrolyzer_Patent_Analysis_Enhanced_Data.csv
✅ Sector summary CSV created: Electrolyzer_Sector_Analysis_Summary.csv

🎯 ENHANCED DOWNLOAD FILES SUMMARY:
📊 Excel Report (Multi-sheet): Electrolyzer_Patent_Analysis_Enhanced_Complete.xlsx
🌐 HTML Report (With charts): Electrolyzer_Patent_Analysis_Enhanced_Report.html
📋 CSV Data File: Electrolyzer_Patent_Analysis_Enhanced_Data.csv
📈 Sector Summary CSV: Electrolyzer_Sector_Analysis_Summary.csv
✨ All files include embedded visualizations and comprehensive analysis
🔗 The HTML file includes interactive Plotly charts
📊 The Excel file contains multiple analysis sheets


In [9]:
# ==============================================
# ANALYSIS COMPLETION SUMMARY
# ==============================================

print("\n" + "=" * 80)
print("🎯 ELECTROLYZER PATENT ANALYSIS - ENHANCED INTERACTIVE ANALYSIS COMPLETE!")
print("=" * 80)

print(f"\n📅 Analysis completed at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"📊 Data source: Electrolyzer Enhanced Final Dataset 2025")
print(f"🔍 Year range analyzed: 2000-2023")
print(f"📝 Analysis type: Enhanced interactive sector analysis with downloadable outputs")

if not raw_data.empty and not applicant_summary.empty:
    print(f"\n📈 KEY RESULTS:")
    print(f"   🏭 Patent families in dataset: {len(electrolyzer_family_ids):,}")
    print(f"   👥 Applicant-patent relationships: {len(raw_data):,}")
    print(f"   🏢 Unique applicant names: {len(applicant_summary):,}")
    print(f"   🏷️ Sectors identified: {applicant_summary['Sector'].nunique()}")
    print(f"   🏆 Top applicant: {applicant_summary.iloc[0]['Applicant_Name']} ({applicant_summary.iloc[0]['Patent_Families']} families)")
    
    print(f"\n✨ ENHANCED FEATURES IMPLEMENTED:")
    print(f"   🎯 Smart sector classification (Database PSN_SECTOR + Enhanced name-based fallback)")
    print(f"   🔍 Special handling for Research Institutions and Academies of Sciences")
    print(f"   🌐 Interactive HTML with pre-loaded search functionality") 
    print(f"   🎨 Color-coded sector visualizations")
    print(f"   📊 Advanced Plotly visualizations (Bar chart, Pie chart, Scatter plot)")
    print(f"   📁 Enhanced downloadable files with embedded diagrams")
    
    print(f"\n📋 SECTOR DISTRIBUTION SUMMARY:")
    sector_summary_final = applicant_summary.groupby('Sector')['Patent_Families'].sum().sort_values(ascending=False)
    total_families_final = sector_summary_final.sum()
    for sector, count in sector_summary_final.items():
        percentage = (count / total_families_final) * 100
        print(f"   🏷️ {sector}: {count:,} families ({percentage:.1f}%)")

    print(f"\n📁 GENERATED FILES:")
    print(f"   🌐 electrolyzer_interactive_analysis.html - Interactive search & aggregation")
    print(f"   📊 Electrolyzer_Patent_Analysis_Enhanced_Complete.xlsx - Multi-sheet Excel report")
    print(f"   🌐 Electrolyzer_Patent_Analysis_Enhanced_Report.html - Comprehensive HTML report with charts")
    print(f"   📋 Electrolyzer_Patent_Analysis_Enhanced_Data.csv - Raw data export")
    print(f"   📈 Electrolyzer_Sector_Analysis_Summary.csv - Sector summary")

else:
    print(f"\n❌ ANALYSIS INCOMPLETE")
    print(f"No electrolyzer patent applicant data could be extracted.")
    print(f"Please check dataset and database connection.")

print(f"\n🔗 USAGE INSTRUCTIONS:")
print(f"   1️⃣ Open 'electrolyzer_interactive_analysis.html' for interactive search")
print(f"   2️⃣ Type keywords like 'TOSHIBA' or 'UNIVERSITY' to filter applicants")
print(f"   3️⃣ Click applicant names to select and see real-time aggregation")
print(f"   4️⃣ Open 'Electrolyzer_Patent_Analysis_Enhanced_Report.html' for comprehensive report")
print(f"   5️⃣ Use Excel file for detailed multi-sheet analysis")

print("\n" + "=" * 80)
print("🚀 ENHANCED ELECTROLYZER PATENT SECTOR ANALYSIS COMPLETE!")
print("=" * 80)
print("\n💡 This analysis provides:")
print("   • Comprehensive sector-based classification of electrolyzer patent applicants")
print("   • Interactive search and aggregation capabilities")
print("   • Advanced visualizations with embedded diagrams") 
print("   • Multiple downloadable formats for further analysis")
print("   • Smart handling of database and name-based sector classification")
print("\n🎉 Ready for use! Open the HTML files to start exploring the interactive features.")


🎯 ELECTROLYZER PATENT ANALYSIS - ENHANCED INTERACTIVE ANALYSIS COMPLETE!

📅 Analysis completed at: 2026-03-12 15:06:14
📊 Data source: Electrolyzer Enhanced Final Dataset 2025
🔍 Year range analyzed: 2000-2023
📝 Analysis type: Enhanced interactive sector analysis with downloadable outputs

📈 KEY RESULTS:
   🏭 Patent families in dataset: 18,811
   👥 Applicant-patent relationships: 36,806
   🏢 Unique applicant names: 12,849
   🏷️ Sectors identified: 6
   🏆 Top applicant: CHINESE ACADEMY OF SCIENCES (308 families)

✨ ENHANCED FEATURES IMPLEMENTED:
   🎯 Smart sector classification (Database PSN_SECTOR + Enhanced name-based fallback)
   🔍 Special handling for Research Institutions and Academies of Sciences
   🌐 Interactive HTML with pre-loaded search functionality
   🎨 Color-coded sector visualizations
   📊 Advanced Plotly visualizations (Bar chart, Pie chart, Scatter plot)
   📁 Enhanced downloadable files with embedded diagrams

📋 SECTOR DISTRIBUTION SUMMARY:
   🏷️ Company: 15,487 families 